# 13.6 Beta 如何影響參考約束與更新？

# 第 13 章：把回答偏好變成訓練

前置：第7章SFT，以及第8–9章的「好回答」規則。可以直接走這條路，不必先學圖片。把兩篇答案像兩道菜一起試喫：喜歡哪一篇，與「兩篇是否都安全」是不同問題。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：試喫兩道菜：比起原廚師，更偏向好回答**

chosen 與 rejected 必須對同一個問題。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/dpo.svg")))

**先想一想：**margin已很大時，把beta加倍，梯度也必定加倍嗎？

beta同時影響loss對margin的敏感度與相對reference的約束解釋。不能把「beta大」簡化成「最終更喜歡chosen」。

**把直覺寫成數學：**導數含beta與sigmoid飽和；不同margin下可觀察非線性。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.alignment import dpo_loss

for beta in (0.1, 1.0, 5.0):
    c = torch.tensor([2.0], requires_grad=True)
    loss = dpo_loss(c, torch.tensor([0.0]), torch.tensor([0.0]), torch.tensor([0.0]), beta)
    loss.backward()
    print(beta, loss.item(), c.grad.item())

**如何讀結果：**這是固定score的局部梯度，不是完整訓練的最終行為；飽和可讓梯度變小。

**只改一件事：**只把margin改成負數。
